In [1]:
from gettext import install
import pip

%pip install requests beautifulsoup4 pandas openpyxl lxml selenium


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import re

def scrape_team_shots_bologna(driver, container, team_name):
    """Scrape all shots for Bologna (tabpanel-left)"""
    print(f"\n{'='*60}")
    print(f"Starting scraping for {team_name}")
    print(f"{'='*60}")
    
    buttons = container.find_elements(By.TAG_NAME, "button")
    print(f"Found {len(buttons)} buttons in container")
    
    if len(buttons) >= 2:
        next_button = buttons[1]
        print("Using second button as next")
    else:
        raise Exception("Could not find next button")
    
    print("\nClicking next to start from the next shot...")
    driver.execute_script("arguments[0].click();", next_button)
    time.sleep(2)
    
    wait = WebDriverWait(driver, 10)
    try:
        wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, "#tabpanel-left > div.d_flex.jc_center.bg_graphics\\.terrain\\.football.mt_xs.bdr-b_lg")
        ))
    except:
        print("Warning: Field section not loaded immediately, waiting...")
        time.sleep(3)
    
    counter_element = container.find_element(By.CSS_SELECTOR, "bdi")
    first_shot_minute = counter_element.text
    
    try:
        shooter_elem = container.find_element(By.CSS_SELECTOR, "a[href*='player']")
        first_shooter = shooter_elem.text.strip()
    except:
        shooter_elem = container.find_element(By.CSS_SELECTOR, "div > div")
        first_shooter = shooter_elem.text.strip().split('\n')[0]
    
    start_section = driver.find_element(By.CSS_SELECTOR,
        "#tabpanel-left > div.d_flex.jc_center.bg_graphics\\.terrain\\.football.mt_xs.bdr-b_lg")
    start_markers = start_section.find_elements(By.CSS_SELECTOR, "circle")
    first_start_x = start_markers[-1].get_attribute('cx') if start_markers else None
    
    first_shot_id = f"{first_shot_minute}|{first_shooter}|{first_start_x}"
    print(f"First shot ID (starting point): {first_shot_id}")
    
    all_shots = []
    iteration = 0
    max_iterations = 50
    
    while iteration < max_iterations:
        iteration += 1
        
        try:
            counter_element = container.find_element(By.CSS_SELECTOR, "bdi")
            current_minute = counter_element.text
            
            shooter_info = None
            try:
                shooter_elem = container.find_element(By.CSS_SELECTOR, "a[href*='player']")
                shooter_info = shooter_elem.text.strip()
            except:
                try:
                    shooter_elem = container.find_element(By.CSS_SELECTOR, "div > div")
                    shooter_info = shooter_elem.text.strip().split('\n')[0]
                except:
                    pass
            
            start_section = driver.find_element(By.CSS_SELECTOR,
                "#tabpanel-left > div.d_flex.jc_center.bg_graphics\\.terrain\\.football.mt_xs.bdr-b_lg")
            start_markers = start_section.find_elements(By.CSS_SELECTOR, "circle")
            
            start_x, start_y = None, None
            if start_markers:
                marker = start_markers[-1]
                start_x = marker.get_attribute('cx')
                start_y = marker.get_attribute('cy')
            
            current_shot_id = f"{current_minute}|{shooter_info}|{start_x}"
            
            if iteration > 1 and current_shot_id == first_shot_id:
                print(f"\nBack to first shot (iteration {iteration}) - stopping")
                break
            
            print(f"\n--- Shot {iteration}: {current_minute} by {shooter_info} ---")
            
            xG = None
            xGOT = None
            outcome = None
            situation = None
            
            try:
                info_div = container.find_element(By.XPATH, "./div[2]")
                additional_info_text = info_div.text.strip()
                lines = additional_info_text.split('\n')
                
                for i, line in enumerate(lines):
                    if line == 'xG' and i+1 < len(lines):
                        xG = lines[i+1]
                    elif line == 'xGOT' and i+1 < len(lines):
                        xGOT = lines[i+1]
                    elif line == 'Outcome' and i+1 < len(lines):
                        outcome = lines[i+1]
                    elif line == 'Situation' and i+1 < len(lines):
                        situation = lines[i+1]
                
                print(f"xG: {xG}, xGOT: {xGOT}, Outcome: {outcome}, Situation: {situation}")
                
            except Exception as e:
                print(f"Could not extract additional info: {e}")
            
            arrival_x, arrival_y = None, None
            
            if outcome and outcome.lower() == 'blocked':
                print("Shot blocked - arrival coordinates set to None")
            else:
                try:
                    arrival_section = driver.find_element(By.CSS_SELECTOR,
                        "#tabpanel-left > div.d_flex.jc_center.mt_sm")
                    g_elements = arrival_section.find_elements(By.CSS_SELECTOR, "g[transform]")
                    
                    for g_elem in g_elements:
                        transform = g_elem.get_attribute('transform')
                        match = re.search(r'translate\(([0-9.]+)[,\s]+([0-9.]+)\)', transform)
                        if match:
                            arrival_x = match.group(1)
                            arrival_y = match.group(2)
                            print(f"Found arrival from transform: ({arrival_x}, {arrival_y})")
                            break
                    
                    if arrival_x is None:
                        arrival_markers = arrival_section.find_elements(By.CSS_SELECTOR, "circle")
                        if arrival_markers:
                            marker = arrival_markers[-1]
                            arrival_x = marker.get_attribute('cx')
                            arrival_y = marker.get_attribute('cy')
                            print(f"Found arrival from circle: ({arrival_x}, {arrival_y})")
                        
                except Exception as e:
                    print(f"Could not extract arrival coordinates: {e}")
            
            shot_data = {
                'team': team_name,
                'shot_number': iteration,
                'minute': current_minute,
                'shooter': shooter_info,
                'xG': xG,
                'xGOT': xGOT,
                'outcome': outcome,
                'situation': situation,
                'start_x': start_x,
                'start_y': start_y,
                'arrival_x': arrival_x,
                'arrival_y': arrival_y
            }
            all_shots.append(shot_data)
            
            print(f"Start: ({start_x}, {start_y}), Arrival: ({arrival_x}, {arrival_y})")
            
            driver.execute_script("arguments[0].click();", next_button)
            time.sleep(1.5)
                
        except Exception as e:
            print(f"Error: {e}")
            import traceback
            traceback.print_exc()
            break
    
    print(f"\nSuccessfully extracted {len(all_shots)} shots for {team_name}")
    return all_shots


def scrape_team_shots_dortmund(driver, team_name):
    """Scrape all shots for Dortmund (tabpanel-right)"""
    print(f"\n{'='*60}")
    print(f"Starting scraping for {team_name}")
    print(f"{'='*60}")
    
    wait = WebDriverWait(driver, 10)
    
    # Click next button to start from first shot
    next_button_xpath = "//*[@id='tabpanel-right']/div[3]/div[1]/button[2]"
    next_button = wait.until(EC.element_to_be_clickable((By.XPATH, next_button_xpath)))
    print("Next button found, clicking to start...")
    driver.execute_script("arguments[0].click();", next_button)
    time.sleep(2)
    
    # Get first shot ID as stopping condition
    info_div = driver.find_element(By.XPATH, "//*[@id='tabpanel-right']/div[3]")
    first_info_text = info_div.text.strip()
    
    start_div = driver.find_element(By.XPATH, "//*[@id='tabpanel-right']/div[2]/div")
    start_circles = start_div.find_elements(By.CSS_SELECTOR, "circle")
    first_start_x = start_circles[-1].get_attribute('cx') if start_circles else None
    
    first_shot_id = f"{first_info_text[:30]}|{first_start_x}"
    print(f"First shot ID (starting point): {first_shot_id}")
    
    all_shots = []
    iteration = 0
    max_iterations = 50
    
    while iteration < max_iterations:
        iteration += 1
        
        try:
            # --- Extract info (minute, shooter, xG, xGOT, outcome, situation) ---
            info_div = driver.find_element(By.XPATH, "//*[@id='tabpanel-right']/div[3]")
            info_text = info_div.text.strip()
            lines = info_text.split('\n')
            
            current_minute = None
            shooter_info = None
            xG = None
            xGOT = None
            outcome = None
            situation = None
            
            for i, line in enumerate(lines):
                if "'" in line and current_minute is None:
                    current_minute = line.strip()
                elif line == 'xG' and i+1 < len(lines):
                    xG = lines[i+1]
                elif line == 'xGOT' and i+1 < len(lines):
                    xGOT = lines[i+1]
                elif line == 'Outcome' and i+1 < len(lines):
                    outcome = lines[i+1]
                elif line == 'Situation' and i+1 < len(lines):
                    situation = lines[i+1]
            
            # Try to get shooter from specific xpath
            try:
                shooter_elem = driver.find_element(By.XPATH, "//*[@id='tabpanel-right']/div[3]/div[1]/div/span")
                shooter_info = shooter_elem.text.strip()
            except:
                pass
            
            # --- Check stopping condition ---
            start_div = driver.find_element(By.XPATH, "//*[@id='tabpanel-right']/div[2]/div")
            start_circles = start_div.find_elements(By.CSS_SELECTOR, "circle")
            
            start_x, start_y = None, None
            if start_circles:
                start_x = start_circles[-1].get_attribute('cx')
                start_y = start_circles[-1].get_attribute('cy')
            
            current_shot_id = f"{info_text[:30]}|{start_x}"
            
            if iteration > 1 and current_shot_id == first_shot_id:
                print(f"\nBack to first shot (iteration {iteration}) - stopping")
                break
            
            print(f"\n--- Shot {iteration}: {current_minute} by {shooter_info} ---")
            print(f"xG: {xG}, xGOT: {xGOT}, Outcome: {outcome}, Situation: {situation}")
            
            # --- Extract arrival coordinates ---
            arrival_x, arrival_y = None, None
            
            if outcome and outcome.lower() == 'blocked':
                print("Shot blocked - arrival coordinates set to None")
            else:
                try:
                    arrival_div = driver.find_element(By.XPATH, "//*[@id='tabpanel-right']/div[1]/div")
                    g_elements = arrival_div.find_elements(By.CSS_SELECTOR, "g[transform]")
                    
                    for g_elem in g_elements:
                        transform = g_elem.get_attribute('transform')
                        match = re.search(r'translate\(([0-9.]+)[,\s]+([0-9.]+)\)', transform)
                        if match:
                            arrival_x = match.group(1)
                            arrival_y = match.group(2)
                            print(f"Found arrival from transform: ({arrival_x}, {arrival_y})")
                            break
                    
                    if arrival_x is None:
                        arrival_circles = arrival_div.find_elements(By.CSS_SELECTOR, "circle")
                        if arrival_circles:
                            arrival_x = arrival_circles[-1].get_attribute('cx')
                            arrival_y = arrival_circles[-1].get_attribute('cy')
                            print(f"Found arrival from circle: ({arrival_x}, {arrival_y})")
                            
                except Exception as e:
                    print(f"Could not extract arrival coordinates: {e}")
            
            print(f"Start: ({start_x}, {start_y}), Arrival: ({arrival_x}, {arrival_y})")
            
            shot_data = {
                'team': team_name,
                'shot_number': iteration,
                'minute': current_minute,
                'shooter': shooter_info,
                'xG': xG,
                'xGOT': xGOT,
                'outcome': outcome,
                'situation': situation,
                'start_x': start_x,
                'start_y': start_y,
                'arrival_x': arrival_x,
                'arrival_y': arrival_y
            }
            all_shots.append(shot_data)
            
            # Click next
            next_button = driver.find_element(By.XPATH, next_button_xpath)
            driver.execute_script("arguments[0].click();", next_button)
            time.sleep(1.5)
        
        except Exception as e:
            print(f"Error: {e}")
            import traceback
            traceback.print_exc()
            break
    
    print(f"\nSuccessfully extracted {len(all_shots)} shots for {team_name}")
    return all_shots


# ─────────────────────────────────────────────
# Main script
# ─────────────────────────────────────────────
driver = webdriver.Chrome()

try:
    url = "https://www.sofascore.com/football/match/bologna-borussia-dortmund/ydbsKdb#id:12764111"
    driver.get(url)
    
    wait = WebDriverWait(driver, 20)
    time.sleep(3)
    
    # ── Loop 1: close the cookie banner if present ──
    cookie_elements = driver.find_elements(By.XPATH, "/html/body/div[4]/div[2]/div[2]/div[2]/div[2]/button[1]")
    if cookie_elements:
        try:
            driver.execute_script("arguments[0].click();", cookie_elements[0])
            print("Cookie banner closed")
            time.sleep(2)
        except Exception as e:
            print(f"Could not close cookie banner: {e}")
    else:
        print("Cookie banner not detected, skipping")
    
    # ── Loop 2: close language pop up if present ──
    language_elements = driver.find_elements(By.XPATH, "//*[@id='portals']/div/div/div/div[1]/button")
    if language_elements:
        try:
            driver.execute_script("arguments[0].click();", language_elements[0])
            print("Language popup closed")
            time.sleep(2)
        except Exception as e:
            print(f"Could not close language popup: {e}")
    else:
        print("Language popup not detected, skipping")
    
    time.sleep(3)
    
    # ── Click Statistics tab ──
    print("Looking for Statistics tab...")
    stats_tab = driver.find_element(By.XPATH, "//a[contains(text(), 'Statistics')]")
    driver.execute_script("arguments[0].scrollIntoView(true);", stats_tab)
    time.sleep(1)
    driver.execute_script("arguments[0].click();", stats_tab)
    print("Clicked on Statistics tab")
    time.sleep(3)
    
    # ── Find container and scrape Bologna ──
    print("\nLooking for shot navigation container...")
    container = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "div.bg-c_surface\\.s2.br_lg.pb_sm.mdDown\\:max-w_\\[312px\\]")
    ))
    print("Container found")
    
    team1_shots = scrape_team_shots_bologna(driver, container, "Bologna")
    
    # ── Switch to Dortmund ──
    print("\n" + "="*60)
    print("Switching to Dortmund...")
    print("="*60)
    
    team2_shots = []
    
    all_buttons = driver.find_elements(By.TAG_NAME, "button")
    team_switch_button = None
    
    for i, btn in enumerate(all_buttons):
        btn_html = btn.get_attribute('outerHTML')
        btn_text = btn.text.strip()
        if 'dortmund' in btn_html.lower() or 'dortmund' in btn_text.lower():
            if 'selected' not in btn.get_attribute('class').lower():
                team_switch_button = btn
                print(f"Found Dortmund button at index {i}")
                break
    
    if not team_switch_button:
        button_containers = driver.find_elements(By.CSS_SELECTOR, "div[class*='horizontal']")
        for container_elem in button_containers:
            buttons_in_container = container_elem.find_elements(By.TAG_NAME, "button")
            if len(buttons_in_container) == 2:
                team_switch_button = buttons_in_container[1]
                print(f"Found second button in container as fallback")
                break
    
    if team_switch_button:
        driver.execute_script("arguments[0].scrollIntoView(true);", team_switch_button)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", team_switch_button)
        print("Clicked on Dortmund button")
        time.sleep(5)
        
        team2_shots = scrape_team_shots_dortmund(driver, "Borussia Dortmund")
    else:
        print("Could not find Dortmund button - saving Bologna data only")
    
    # ── Combine and show results ──
    all_shots = team1_shots + team2_shots
    
    if all_shots:
        df = pd.DataFrame(all_shots)

        # Substitute '-' with None in the xGOT column
        df['xGOT'] = df['xGOT'].replace('-', None)

        # Normalize accented characters in shooter names
        df['shooter'] = df['shooter'].str.replace('ī', 'i', regex=False)
        
        print(f"\n{'='*60}")
        print(f"FINAL RESULTS")
        print(f"{'='*60}")
        print(f"Total shots extracted: {len(all_shots)}")
        print(f"Bologna shots: {len(team1_shots)}")
        print(f"Borussia Dortmund shots: {len(team2_shots)}")
        print(df.head(30))
        
        # Export to CSV
        #df.to_csv('shotmap_bologna_dortmund.csv', index=False)
        print("\nComplete shot map saved to 'shotmap_bologna_dortmund.csv'")
    else:
        print("\nNo shots extracted")

finally:
    driver.quit()

Cookie banner closed
Language popup closed
Looking for Statistics tab...


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"//a[contains(text(), 'Statistics')]"}
  (Session info: chrome=148.0.7778.217); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff733b47de5+14895]
	chromedriver!GetHandleVerifier [0x7ff733b47e50+14900]
	chromedriver!(No symbol) [0x7ff7338ad5ad]
	chromedriver!(No symbol) [0x7ff733907822]
	chromedriver!(No symbol) [0x7ff733907b2c]
	chromedriver!(No symbol) [0x7ff733957d17]
	chromedriver!(No symbol) [0x7ff73395486f]
	chromedriver!(No symbol) [0x7ff7338f9df8]
	chromedriver!(No symbol) [0x7ff7338face3]
	chromedriver!GetHandleVerifier [0x7ff733e5cc49+3296f9]
	chromedriver!GetHandleVerifier [0x7ff733e57375+323e25]
	chromedriver!GetHandleVerifier [0x7ff733e7bc82+348732]
	chromedriver!GetHandleVerifier [0x7ff733b66045+32af5]
	chromedriver!GetHandleVeri